# Maestro: entrenamiento en Poisson y evaluación en ABIDES (BORRADOR de PS para MR)

1. Entrena el Maestro **en Poisson** (una jornada < 2 s) con impacto acumulado entre ventanas y Ejecutores fijos sin entrenar. Guarda el Maestro y los Ejecutores que usó.
2. Evalúa **en ABIDES**, en las mismas 2 meta-órdenes y 2 semillas y con el mismo impacto: el Maestro entrenado, el mismo Maestro sin entrenar, y TWAP/VWAP.

Es una prueba de que todo el camino funciona en ABIDES, no la evaluación final (esa son las 90 corridas).

**Cómo usarlo:** «Ejecutar todas». Dura unas **2 horas y 30 minutos** (≈ 25 min de entrenamiento y ≈ 6 min por jornada en ABIDES). Si se corta, vuelve a ejecutar todo: retoma donde quedó.

Todo queda en **`slippage_colab/resultados/maestro_impacto`** y se arma `maestro_impacto.zip`.

## Preparación del entorno (celdas 1 a 9)

In [ ]:
# CELDA 1 (reinicia el entorno solo; el aviso "sesión fallada" es normal, no la repitas)
!pip install -q "condacolab<0.2"
import condacolab
condacolab.install()

In [ ]:
# CELDA 2
import os
if not os.path.exists('/usr/local/envs/abides'):
    !conda create -n abides python=3.9 -y -q
!conda run -n abides python --version

In [ ]:
# CELDA 3 (unos minutos; un aviso de Pillow es normal)
%env PIP_DISABLE_PIP_VERSION_CHECK=1
!conda run -n abides python -m pip install -q pip==23.3.2 setuptools==57.5.0 wheel==0.38.4
!conda run -n abides python -m pip install -q "cython<3" numpy==1.22.0
!conda run -n abides python -m pip install -q --no-build-isolation --no-deps gym==0.18.0
!conda run -n abides python -m pip install -q --no-build-isolation numpy==1.22.0 "cloudpickle>=1.2,<1.7" "pyglet>=1.4,<=1.5.0" "Pillow<10" pandas==1.2.4 scipy==1.10.0 coloredlogs==15.0.1 psutil==5.8.0 tqdm==4.61.1 pomegranate==0.14.5 "ray[tune]==1.7.0" "protobuf==3.20.3" pyarrow==11.0.0 matplotlib==3.5.3 pytest
!conda run -n abides python -m pip install -q "gymnasium==0.29.1" "importlib-metadata>=4.8" farama-notifications --no-deps

In [ ]:
# CELDA 4 (deben salir tres "OK")
import os
if not os.path.exists('/content/abides-jpmc-public'):
    !git clone -q https://github.com/jpmorganchase/abides-jpmc-public.git /content/abides-jpmc-public
for pkg in ['abides-core', 'abides-markets', 'abides-gym']:
    !cd /content/abides-jpmc-public/{pkg} && conda run -n abides python setup.py -q install > /content/install_{pkg}.log 2>&1 && echo "{pkg} OK" || (echo "{pkg} FALLO"; tail -20 /content/install_{pkg}.log)

In [ ]:
# CELDA 5 (debe imprimir 0.18.0 1.22.0 1.2.4 0.29.1 1.7.0)
!conda run -n abides python -c "import gym, numpy, pandas, gymnasium, ray, abides_core, abides_markets, abides_gym; print(gym.__version__, numpy.__version__, pandas.__version__, gymnasium.__version__, ray.__version__)"

In [ ]:
# CELDA 6 (torch; debe imprimir torch 1.13.1+cpu | numpy 1.22.0)
!conda run -n abides python -m pip install -q torch==1.13.1+cpu --extra-index-url https://download.pytorch.org/whl/cpu
!conda run -n abides python -c "import torch, numpy; print('torch', torch.__version__, '| numpy', numpy.__version__)"

In [ ]:
# CELDA 7 (código del proyecto: rama feature/ps-borrador-mr-maestro; debe mostrar un commit y un número mayor que 0)
import os
if not os.path.exists('/content/slippage'):
    !git clone -q https://github.com/Vatelfil/slippage.git /content/slippage
!cd /content/slippage && git fetch -q origin && git checkout -q -f origin/feature/ps-borrador-mr-maestro
!cd /content/slippage && git log --oneline -1 && grep -c "train_maestro" src/training/maestro_trainer.py


In [ ]:
# CELDA 8 (conecta Drive; acepta el permiso)
from google.colab import drive
drive.mount('/content/drive')
RESULTS_DIR = '/content/drive/MyDrive/slippage_colab/resultados'
import os; os.makedirs(RESULTS_DIR, exist_ok=True)
print(RESULTS_DIR)

In [ ]:
# CELDA 9 (seaborn, que necesita el módulo de benchmarks)
!conda run -n abides python -m pip install -q seaborn==0.12.2 --no-deps

## Paso 1: entrenar el Maestro en Poisson (≈ 25 min)

In [ ]:
# 1.000 jornadas, impacto acumulado (Y = 0,5, tau = 120 min), Ejecutores fijos sin entrenar
!cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/entrenar_maestro.py --jornadas 1000 --impacto --out-dir {RESULTS_DIR}/maestro_impacto 2>&1 | grep -v UserWarning | tail -25


## Paso 2: evaluar en ABIDES (≈ 2 h)

In [ ]:
# Maestro entrenado (mismos Ejecutores con que aprendió)
!cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/colab/evaluar_sistema_ps.py --meta-ordenes data/meta_ordenes/meta_ordenes_v2_candidata.json --entorno abides --ordenes MO-01 MO-02 --seeds 2 --impacto --out-dir {RESULTS_DIR}/maestro_impacto/eval_abides --max-minutes 150 --modo muestreo --ckpt-maestro {RESULTS_DIR}/maestro_impacto/checkpoint_last.pt --ckpt-ejecutor apertura={RESULTS_DIR}/maestro_impacto/ejecutor_apertura.pt --ckpt-ejecutor media_jornada={RESULTS_DIR}/maestro_impacto/ejecutor_media_jornada.pt --ckpt-ejecutor cierre={RESULTS_DIR}/maestro_impacto/ejecutor_cierre.pt --nombre PPO_maestro 2>&1 | grep -v UserWarning


In [ ]:
# El mismo Maestro sin entrenar (control), con los mismos Ejecutores
!cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/colab/evaluar_sistema_ps.py --meta-ordenes data/meta_ordenes/meta_ordenes_v2_candidata.json --entorno abides --ordenes MO-01 MO-02 --seeds 2 --impacto --out-dir {RESULTS_DIR}/maestro_impacto/eval_abides --max-minutes 150 --modo muestreo --ckpt-ejecutor apertura={RESULTS_DIR}/maestro_impacto/ejecutor_apertura.pt --ckpt-ejecutor media_jornada={RESULTS_DIR}/maestro_impacto/ejecutor_media_jornada.pt --ckpt-ejecutor cierre={RESULTS_DIR}/maestro_impacto/ejecutor_cierre.pt --nombre PPO_maestro_sin_entrenar 2>&1 | grep -v UserWarning


In [ ]:
# TWAP y VWAP con el mismo impacto
!cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/colab/benchmarks_meta_ordenes_ps.py --meta-ordenes data/meta_ordenes/meta_ordenes_v2_candidata.json --entorno abides --ordenes MO-01 MO-02 --seeds 2 --impacto --out-dir {RESULTS_DIR}/maestro_impacto/eval_abides --max-minutes 150 2>&1 | grep -v UserWarning


In [ ]:
# Tabla con las cuatro estrategias
!cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/consolidar_resultados_3_2_2.py --entradas {RESULTS_DIR}/maestro_impacto/eval_abides --sistema PPO_maestro --out-dir {RESULTS_DIR}/maestro_impacto/tabla
print(open(f'{RESULTS_DIR}/maestro_impacto/tabla/tabla_maestra.md', encoding='utf-8').read())


## Zip para descargar

In [ ]:
# CELDA FINAL: zip pequeño (sin checkpoints) para pasárselo a PS
import os, zipfile
base = f'{RESULTS_DIR}/maestro_impacto'
archivos = [os.path.join(c, a) for c, _, fs in os.walk(base) if 'checkpoints' not in c for a in fs if a.endswith(('.json', '.csv', '.md'))]
print(len(archivos), 'archivos')
zpath = f'{RESULTS_DIR}/maestro_impacto.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for a in archivos:
        z.write(a, os.path.relpath(a, RESULTS_DIR))
print('zip listo:', zpath, '-> descárgalo desde slippage_colab/resultados en Drive')
